In [ ]:
import uproot
import awkward as ak
import numpy as np
import sklearn.metrics as m
import boost_histogram as bh
import glob
import os

from matplotlib import pyplot as plt
import matplotlib as mpl
from cycler import cycler
import mplhep as hep

use_helvet = False  ## true: use helvetica for plots, make sure the system have the font installed
if use_helvet:
    CMShelvet = hep.style.CMS
    CMShelvet["font.sans-serif"] = ["Helvetica", "Arial"]
    plt.style.use(CMShelvet)
else:
    plt.style.use(hep.style.CMS)

plt.style.use(hep.style.CMS)

base = 10

In [ ]:
## Mass decorelation
import boost_histogram as bh


def plot_mass_decorr_ptbinned(
    sam,
    taggername,
    sname,
    bname,
    mistags=[1, 0.01],
    pt_edges=[200, 400, 600, 1000, 1500, 2000, 3000],
):
    colorlist = ["blue", "red", "green", "cyan", "darkorange", "magenta"]
    plot_kwargs_list = [
        dict(
            color="black",
            linestyle="none",
            marker="o",
            markersize=8,
            markeredgecolor="black",
            markerfacecolor="white",
        ),
        dict(
            color="#5790fc",
            linestyle="none",
            marker="o",
            markersize=6,
            markeredgecolor="#5790fc",
            markerfacecolor="#5790fc",
        ),
        dict(
            color="red",
            linestyle="none",
            marker="s",
            markersize=8,
            markeredgecolor="red",
            markerfacecolor="white",
        ),
        dict(
            color="darkorange",
            linestyle="none",
            marker="s",
            markersize=8,
            markeredgecolor="darkorange",
            markerfacecolor="darkorange",
        ),
    ]
    # plot_kwargs_list = [
    #     dict(color='black', linestyle='none', marker='o', markersize=8, markeredgecolor='black', markerfacecolor='white'),
    #     dict(color='darkred', linestyle='none', marker='o', markersize=6, markeredgecolor='darkred', markerfacecolor='darkred'),
    #     dict(color='red', linestyle='none', marker='s', markersize=8, markeredgecolor='red', markerfacecolor='white'),
    #     dict(color='darkorange', linestyle='none', marker='s', markersize=8, markeredgecolor='darkorange', markerfacecolor='darkorange'),
    # ]
    mpl.rcParams["axes.prop_cycle"] = cycler(color=colorlist)

    sam_title_map = {
        "ttbar": r"t$\overline{t}$ sample",
        "qcd": r"QCD multijet",
        "qcd-xsecw": r"QCD multijet",
    }
    label_dict = {
        "QCD": "QCD",
        "HWWQQQQ": r"H$\rightarrow$WW 4q",
        "HWWQQQ": r"H$\rightarrow$WW 3q",
        "HWWQQev": r"H$\rightarrow$WW $e\nu qq$",
        "HWWQQmv": r"H$\rightarrow$WW $\mu\nu qq$",
        "HWWQQta": r"H$\rightarrow$WW $\tau\nu qq$",
        "HWWQQtauev": r"H$\rightarrow$WW $\tau_e\nu qq$",
        "HWWQQtaumv": r"H$\rightarrow$WW $\tau_\mu\nu qq$",
        "HWWQQtauhv": r"H$\rightarrow$WW $\tau_h\nu qq$",
        "HIGbb": r"H$\rightarrow$bb",
        "HIGcc": r"H$\rightarrow$cc",
        "HIGss": r"H$\rightarrow$ss",
        "HIGqq": r"H$\rightarrow$qq",
        "HIGbc": r"$H^\pm \rightarrow$bc",
        "HIGcs": r"$H^\pm \rightarrow$cs",
        "HIGbs": r"$H\rightarrow$bs",
        "HIGgg": r"$H\rightarrow$gg",
        "HIGee": r"$H\rightarrow$ee",
        "HIGmm": r"$H\rightarrow \mu\mu$",
        "HIGtauhvtauev": r"H$\rightarrow\tau_h\tau_e$",
        "HIGtauhvtaumv": r"H$\rightarrow\tau_h\tau_\mu$",
        "HIGtauhvtauhv": r"H$\rightarrow\tau_h\tau_h$",
        "HIGll": r"H$\rightarrow$qq (q=u/d/s)",
        "HIGtauhtaue": r"H$\rightarrow\tau_h\tau_e$",
        "HIGtauhtaum": r"H$\rightarrow\tau_h\tau_\mu$",
        "HIGtauhtauh": r"H$\rightarrow\tau_h\tau_h$",
        "HIGGS2Pbb": r"H$\rightarrow$bb",
        "HIGGS2Pcc": r"H$\rightarrow$cc",
        "HIGGS2Pss": r"H$\rightarrow$ss",
        "HIGGS2Pqq": r"H$\rightarrow$qq",
        "HIGGS2Pbc": r"$H^\pm \rightarrow$bc",
        "HIGGS2Pcs": r"$H^\pm \rightarrow$cs",
        "HIGGS2Pbs": r"$H\rightarrow$bs",
        "HIGGS2Pgg": r"$H\rightarrow$gg",
        "HIGGS2Pee": r"$H\rightarrow$ee",
        "HIGGS2Pmm": r"$H\rightarrow \mu\mu$",
        "HIGGS2Ptauhvtauev": r"H$\rightarrow\tau_h\tau_e$",
        "HIGGS2Ptauhvtaumv": r"H$\rightarrow\tau_h\tau_\mu$",
        "HIGGS2Ptauhvtauhv": r"H$\rightarrow\tau_h\tau_h$",
        "HIGGS2Ptauhtaue": r"H$\rightarrow\tau_h\tau_e$",
        "HIGGS2Ptauhtaum": r"H$\rightarrow\tau_h\tau_\mu$",
        "HIGGS2Ptauhtauh": r"H$\rightarrow\tau_h\tau_h$",
        "TTBARbWall": r"t$\rightarrow$bW all",
        "TTBARbWhad": r"t$\rightarrow$bW had.",
        "TTBARbWQQ": r"t$\rightarrow bW \rightarrow bqq$",
        "TTBARbWev": r"t$\rightarrow bW \rightarrow be\nu$",
        "TTBARbWmv": r"t$\rightarrow bW \rightarrow b\mu\nu$",
        "TTBARbWta": r"t$\rightarrow bW \rightarrow b\tau\nu$",
        "TTBARbWtauhv": r"t$\rightarrow bW \rightarrow b\tau_h\nu$",
        "QCDAndTTBARbWall": r"QCD + t$\rightarrow$bW all",
        "TTBARINCLall": r"t$\rightarrow$bW all (incl.)",
        "TTBARINCLhad": r"t$\rightarrow$bW had. (incl.)",
        "TTBARINCLel": r"t$\rightarrow bW \rightarrow be\nu$ (incl.)",
        "TTBARINCLmu": r"t$\rightarrow bW \rightarrow b\mu\nu$ (incl.)",
        "SMTTBARSLbWall": r"t$\rightarrow$bW all",
        "SMTTBARSLbWhad": r"t$\rightarrow$bW had.",
        "SMTTBARSLbWQQ": r"t$\rightarrow bW \rightarrow bqq$",
        "SMTTBARSLbWev": r"t$\rightarrow bW \rightarrow be\nu$",
        "SMTTBARSLbWmv": r"t$\rightarrow bW \rightarrow b\mu\nu$",
        "SMTTBARSLbWta": r"t$\rightarrow bW \rightarrow b\tau\nu$",
        "SMTTBARSLbWcs": r"t$\rightarrow bW \rightarrow bcs$",
        "SMTTBARSLbWqq": r"t$\rightarrow bW \rightarrow bqq$",
        "XBCbc": r"$H\prime\rightarrow$bc",
        "Wcs": r"W$\rightarrow$cs",
        "Wqq": r"W$\rightarrow$qq",
        "Zbb": r"Z$\rightarrow$bb",
        "Zcc": r"Z$\rightarrow$cc",
        "Zqq": r"Z$\rightarrow$qq",
        "Zss": r"Z$\rightarrow$ss",
        "Zll": r"Z$\rightarrow$qq (q=u/d/s)",
        "WQQ": r"W$\rightarrow$qq (all)",
        "ZQQ": r"Z$\rightarrow$qq (all)",
    }
    variable_dict = {
        "QCD": ["label_QCD_bb", "label_QCD_cc", "label_QCD_b", "label_QCD_c", "label_QCD_others"],
        **{
            f"HIG{n}": [f"label_H_{n}"]
            for n in [
                "bb",
                "cc",
                "qq",
                "ss",
                "bc",
                "cs",
                "bs",
                "gg",
                "ee",
                "mm",
                "tauhtaue",
                "tauhtaum",
                "tauhtauh",
            ]
        },
        **{
            f"HIGGS2P{n}": [f"label_H_{n}"]
            for n in [
                "bb",
                "cc",
                "qq",
                "ss",
                "bc",
                "cs",
                "bs",
                "gg",
                "ee",
                "mm",
                "tauhtaue",
                "tauhtaum",
                "tauhtauh",
            ]
        },
        "HIGll": [f"label_H_{n}" for n in ["ss", "qq"]],
        **{
            f"HWW{n}": [f"label_H_WW_{n}"]
            for n in [
                "cscs",
                "csqq",
                "qqqq",
                "csc",
                "css",
                "csq",
                "qqc",
                "qqs",
                "qqq",
                "csev",
                "qqev",
                "csmv",
                "qqmv",
                "cstauev",
                "qqtauev",
                "cstaumv",
                "qqtaumv",
                "cstauhv",
                "qqtauhv",
            ]
        },
        **{
            f"TTBAR{n}": [f"label_Top_{n}"]
            for n in [
                "bWcs",
                "bWqq",
                "bWc",
                "bWs",
                "bWq",
                "bWev",
                "bWmv",
                "bWtauev",
                "bWtaumv",
                "bWtauhv",
                "Wcs",
                "Wqq",
                "Wev",
                "Wmv",
                "Wtauev",
                "Wtaumv",
                "Wtauhv",
            ]
        },
        **{f"W{n}": [f"label_H_{n}"] for n in ["qq", "cs"]},
        **{f"Z{n}": [f"label_H_{n}"] for n in ["bb", "cc", "qq", "ss"]},
        "HWWQQQQ": [f"label_H_WW_{n}" for n in ["cscs", "csqq", "qqqq"]],
        "HWWQQQ": [f"label_H_WW_{n}" for n in ["csc", "css", "csq", "qqc", "qqs", "qqq"]],
        "HWWQQev": [f"label_H_WW_{n}" for n in ["csev", "qqev"]],
        "HWWQQmv": [f"label_H_WW_{n}" for n in ["csmv", "qqmv"]],
        "HWWQQtauev": [f"label_H_WW_{n}" for n in ["cstauev", "qqtauev"]],
        "HWWQQtaumv": [f"label_H_WW_{n}" for n in ["cstaumv", "qqtaumv"]],
        "HWWQQtauhv": [f"label_H_WW_{n}" for n in ["cstauhv", "qqtauhv"]],
        "HWWQQta": [
            f"label_H_WW_{n}"
            for n in ["cstauev", "qqtauev", "cstaumv", "qqtaumv", "cstauhv", "qqtauhv"]
        ],
        # 'HWWel': [f'label_H_WW_{n}' for n in ['csev', 'cstauev', 'qqev', 'qqtauev']],
        # 'HWWmu': [f'label_H_WW_{n}' for n in ['csmv', 'cstaumv', 'qqmv', 'qqtaumv']],
        "TTBARbWall": [
            f"label_Top_{n}"
            for n in [
                "bWcs",
                "bWqq",
                "bWc",
                "bWs",
                "bWq",
                "bWev",
                "bWmv",
                "bWtauev",
                "bWtaumv",
                "bWtauhv",
            ]
        ],
        "TTBARbWhad": [f"label_Top_{n}" for n in ["bWcs", "bWqq", "bWc", "bWs", "bWq"]],
        "TTBARbWQQ": [f"label_Top_{n}" for n in ["bWcs", "bWqq"]],
        "TTBARbWta": [f"label_Top_{n}" for n in ["bWtauev", "bWtaumv", "bWtauhv"]],
        "TTBARbWtauhv": [f"label_Top_{n}" for n in ["bWtauhv"]],
        "TTBARall": [
            f"label_Top_{n}"
            for n in [
                "bWcs",
                "bWqq",
                "bWc",
                "bWs",
                "bWq",
                "bWev",
                "bWmv",
                "bWtauev",
                "bWtaumv",
                "bWtauhv",
                "Wcs",
                "Wqq",
                "Wev",
                "Wmv",
                "Wtauev",
                "Wtaumv",
                "Wtauhv",
            ]
        ],
        "QCDAndTTBARbWall": [
            "label_QCD_bb",
            "label_QCD_cc",
            "label_QCD_b",
            "label_QCD_c",
            "label_QCD_others",
        ]
        + [
            f"label_Top_{n}"
            for n in [
                "bWcs",
                "bWqq",
                "bWc",
                "bWs",
                "bWq",
                "bWev",
                "bWmv",
                "bWtauev",
                "bWtaumv",
                "bWtauhv",
            ]
        ],
        "WQQ": [f"label_H_{n}" for n in ["qq", "cs"]],
        "ZQQ": [f"label_H_{n}" for n in ["bb", "cc", "ss", "qq"]],
        "Zll": [f"label_H_{n}" for n in ["ss", "qq"]],
    }
    label_list = [
        "label_Top_bWcs",
        "label_Top_bWqq",
        "label_Top_bWc",
        "label_Top_bWs",
        "label_Top_bWq",
        "label_Top_bWev",
        "label_Top_bWmv",
        "label_Top_bWtauev",
        "label_Top_bWtaumv",
        "label_Top_bWtauhv",
        "label_Top_Wcs",
        "label_Top_Wqq",
        "label_Top_Wev",
        "label_Top_Wmv",
        "label_Top_Wtauev",
        "label_Top_Wtaumv",
        "label_Top_Wtauhv",
        "label_H_bb",
        "label_H_cc",
        "label_H_ss",
        "label_H_qq",
        "label_H_bc",
        "label_H_bs",
        "label_H_cs",
        "label_H_gg",
        "label_H_ee",
        "label_H_mm",
        "label_H_tauhtaue",
        "label_H_tauhtaum",
        "label_H_tauhtauh",
        "label_H_WW_cscs",
        "label_H_WW_csqq",
        "label_H_WW_qqqq",
        "label_H_WW_csc",
        "label_H_WW_css",
        "label_H_WW_csq",
        "label_H_WW_qqc",
        "label_H_WW_qqs",
        "label_H_WW_qqq",
        "label_H_WW_csev",
        "label_H_WW_qqev",
        "label_H_WW_csmv",
        "label_H_WW_qqmv",
        "label_H_WW_cstauev",
        "label_H_WW_qqtauev",
        "label_H_WW_cstaumv",
        "label_H_WW_qqtaumv",
        "label_H_WW_cstauhv",
        "label_H_WW_qqtauhv",
        "label_H_WxWx_cscs",
        "label_H_WxWx_csqq",
        "label_H_WxWx_qqqq",
        "label_H_WxWx_csc",
        "label_H_WxWx_css",
        "label_H_WxWx_csq",
        "label_H_WxWx_qqc",
        "label_H_WxWx_qqs",
        "label_H_WxWx_qqq",
        "label_H_WxWx_csev",
        "label_H_WxWx_qqev",
        "label_H_WxWx_csmv",
        "label_H_WxWx_qqmv",
        "label_H_WxWx_cstauev",
        "label_H_WxWx_qqtauev",
        "label_H_WxWx_cstaumv",
        "label_H_WxWx_qqtaumv",
        "label_H_WxWx_cstauhv",
        "label_H_WxWx_qqtauhv",
        "label_H_WxWxStar_cscs",
        "label_H_WxWxStar_csqq",
        "label_H_WxWxStar_qqqq",
        "label_H_WxWxStar_csc",
        "label_H_WxWxStar_css",
        "label_H_WxWxStar_csq",
        "label_H_WxWxStar_qqc",
        "label_H_WxWxStar_qqs",
        "label_H_WxWxStar_qqq",
        "label_H_WxWxStar_csev",
        "label_H_WxWxStar_qqev",
        "label_H_WxWxStar_csmv",
        "label_H_WxWxStar_qqmv",
        "label_H_WxWxStar_cstauev",
        "label_H_WxWxStar_qqtauev",
        "label_H_WxWxStar_cstaumv",
        "label_H_WxWxStar_qqtaumv",
        "label_H_WxWxStar_cstauhv",
        "label_H_WxWxStar_qqtauhv",
        "label_H_ZZ_bbbb",
        "label_H_ZZ_bbcc",
        "label_H_ZZ_bbss",
        "label_H_ZZ_bbqq",
        "label_H_ZZ_cccc",
        "label_H_ZZ_ccss",
        "label_H_ZZ_ccqq",
        "label_H_ZZ_ssss",
        "label_H_ZZ_ssqq",
        "label_H_ZZ_qqqq",
        "label_H_ZZ_bbb",
        "label_H_ZZ_bbc",
        "label_H_ZZ_bbs",
        "label_H_ZZ_bbq",
        "label_H_ZZ_ccb",
        "label_H_ZZ_ccc",
        "label_H_ZZ_ccs",
        "label_H_ZZ_ccq",
        "label_H_ZZ_ssb",
        "label_H_ZZ_ssc",
        "label_H_ZZ_sss",
        "label_H_ZZ_ssq",
        "label_H_ZZ_qqb",
        "label_H_ZZ_qqc",
        "label_H_ZZ_qqs",
        "label_H_ZZ_qqq",
        "label_H_ZZ_bbee",
        "label_H_ZZ_bbmm",
        "label_H_ZZ_bbe",
        "label_H_ZZ_bbm",
        "label_H_ZZ_bee",
        "label_H_ZZ_bmm",
        "label_H_ZZ_bbtauhtaue",
        "label_H_ZZ_bbtauhtaum",
        "label_H_ZZ_bbtauhtauh",
        "label_H_ZZ_btauhtaue",
        "label_H_ZZ_btauhtaum",
        "label_H_ZZ_btauhtauh",
        "label_H_ZZ_ccee",
        "label_H_ZZ_ccmm",
        "label_H_ZZ_cce",
        "label_H_ZZ_ccm",
        "label_H_ZZ_cee",
        "label_H_ZZ_cmm",
        "label_H_ZZ_cctauhtaue",
        "label_H_ZZ_cctauhtaum",
        "label_H_ZZ_cctauhtauh",
        "label_H_ZZ_ctauhtaue",
        "label_H_ZZ_ctauhtaum",
        "label_H_ZZ_ctauhtauh",
        "label_H_ZZ_ssee",
        "label_H_ZZ_ssmm",
        "label_H_ZZ_sse",
        "label_H_ZZ_ssm",
        "label_H_ZZ_see",
        "label_H_ZZ_smm",
        "label_H_ZZ_sstauhtaue",
        "label_H_ZZ_sstauhtaum",
        "label_H_ZZ_sstauhtauh",
        "label_H_ZZ_stauhtaue",
        "label_H_ZZ_stauhtaum",
        "label_H_ZZ_stauhtauh",
        "label_H_ZZ_qqee",
        "label_H_ZZ_qqmm",
        "label_H_ZZ_qqe",
        "label_H_ZZ_qqm",
        "label_H_ZZ_qee",
        "label_H_ZZ_qmm",
        "label_H_ZZ_qqtauhtaue",
        "label_H_ZZ_qqtauhtaum",
        "label_H_ZZ_qqtauhtauh",
        "label_H_ZZ_qtauhtaue",
        "label_H_ZZ_qtauhtaum",
        "label_H_ZZ_qtauhtauh",
        "label_H_ZxZx_bbbb",
        "label_H_ZxZx_bbcc",
        "label_H_ZxZx_bbss",
        "label_H_ZxZx_bbqq",
        "label_H_ZxZx_cccc",
        "label_H_ZxZx_ccss",
        "label_H_ZxZx_ccqq",
        "label_H_ZxZx_ssss",
        "label_H_ZxZx_ssqq",
        "label_H_ZxZx_qqqq",
        "label_H_ZxZx_bbb",
        "label_H_ZxZx_bbc",
        "label_H_ZxZx_bbs",
        "label_H_ZxZx_bbq",
        "label_H_ZxZx_ccb",
        "label_H_ZxZx_ccc",
        "label_H_ZxZx_ccs",
        "label_H_ZxZx_ccq",
        "label_H_ZxZx_ssb",
        "label_H_ZxZx_ssc",
        "label_H_ZxZx_sss",
        "label_H_ZxZx_ssq",
        "label_H_ZxZx_qqb",
        "label_H_ZxZx_qqc",
        "label_H_ZxZx_qqs",
        "label_H_ZxZx_qqq",
        "label_H_ZxZx_bbee",
        "label_H_ZxZx_bbmm",
        "label_H_ZxZx_bbe",
        "label_H_ZxZx_bbm",
        "label_H_ZxZx_bee",
        "label_H_ZxZx_bmm",
        "label_H_ZxZx_bbtauhtaue",
        "label_H_ZxZx_bbtauhtaum",
        "label_H_ZxZx_bbtauhtauh",
        "label_H_ZxZx_btauhtaue",
        "label_H_ZxZx_btauhtaum",
        "label_H_ZxZx_btauhtauh",
        "label_H_ZxZx_ccee",
        "label_H_ZxZx_ccmm",
        "label_H_ZxZx_cce",
        "label_H_ZxZx_ccm",
        "label_H_ZxZx_cee",
        "label_H_ZxZx_cmm",
        "label_H_ZxZx_cctauhtaue",
        "label_H_ZxZx_cctauhtaum",
        "label_H_ZxZx_cctauhtauh",
        "label_H_ZxZx_ctauhtaue",
        "label_H_ZxZx_ctauhtaum",
        "label_H_ZxZx_ctauhtauh",
        "label_H_ZxZx_ssee",
        "label_H_ZxZx_ssmm",
        "label_H_ZxZx_sse",
        "label_H_ZxZx_ssm",
        "label_H_ZxZx_see",
        "label_H_ZxZx_smm",
        "label_H_ZxZx_sstauhtaue",
        "label_H_ZxZx_sstauhtaum",
        "label_H_ZxZx_sstauhtauh",
        "label_H_ZxZx_stauhtaue",
        "label_H_ZxZx_stauhtaum",
        "label_H_ZxZx_stauhtauh",
        "label_H_ZxZx_qqee",
        "label_H_ZxZx_qqmm",
        "label_H_ZxZx_qqe",
        "label_H_ZxZx_qqm",
        "label_H_ZxZx_qee",
        "label_H_ZxZx_qmm",
        "label_H_ZxZx_qqtauhtaue",
        "label_H_ZxZx_qqtauhtaum",
        "label_H_ZxZx_qqtauhtauh",
        "label_H_ZxZx_qtauhtaue",
        "label_H_ZxZx_qtauhtaum",
        "label_H_ZxZx_qtauhtauh",
        "label_H_ZxZxStar_bbbb",
        "label_H_ZxZxStar_bbcc",
        "label_H_ZxZxStar_bbss",
        "label_H_ZxZxStar_bbqq",
        "label_H_ZxZxStar_cccc",
        "label_H_ZxZxStar_ccss",
        "label_H_ZxZxStar_ccqq",
        "label_H_ZxZxStar_ssss",
        "label_H_ZxZxStar_ssqq",
        "label_H_ZxZxStar_qqqq",
        "label_H_ZxZxStar_bbb",
        "label_H_ZxZxStar_bbc",
        "label_H_ZxZxStar_bbs",
        "label_H_ZxZxStar_bbq",
        "label_H_ZxZxStar_ccb",
        "label_H_ZxZxStar_ccc",
        "label_H_ZxZxStar_ccs",
        "label_H_ZxZxStar_ccq",
        "label_H_ZxZxStar_ssb",
        "label_H_ZxZxStar_ssc",
        "label_H_ZxZxStar_sss",
        "label_H_ZxZxStar_ssq",
        "label_H_ZxZxStar_qqb",
        "label_H_ZxZxStar_qqc",
        "label_H_ZxZxStar_qqs",
        "label_H_ZxZxStar_qqq",
        "label_H_ZxZxStar_bbee",
        "label_H_ZxZxStar_bbmm",
        "label_H_ZxZxStar_bbe",
        "label_H_ZxZxStar_bbm",
        "label_H_ZxZxStar_bee",
        "label_H_ZxZxStar_bmm",
        "label_H_ZxZxStar_bbtauhtaue",
        "label_H_ZxZxStar_bbtauhtaum",
        "label_H_ZxZxStar_bbtauhtauh",
        "label_H_ZxZxStar_btauhtaue",
        "label_H_ZxZxStar_btauhtaum",
        "label_H_ZxZxStar_btauhtauh",
        "label_H_ZxZxStar_ccee",
        "label_H_ZxZxStar_ccmm",
        "label_H_ZxZxStar_cce",
        "label_H_ZxZxStar_ccm",
        "label_H_ZxZxStar_cee",
        "label_H_ZxZxStar_cmm",
        "label_H_ZxZxStar_cctauhtaue",
        "label_H_ZxZxStar_cctauhtaum",
        "label_H_ZxZxStar_cctauhtauh",
        "label_H_ZxZxStar_ctauhtaue",
        "label_H_ZxZxStar_ctauhtaum",
        "label_H_ZxZxStar_ctauhtauh",
        "label_H_ZxZxStar_ssee",
        "label_H_ZxZxStar_ssmm",
        "label_H_ZxZxStar_sse",
        "label_H_ZxZxStar_ssm",
        "label_H_ZxZxStar_see",
        "label_H_ZxZxStar_smm",
        "label_H_ZxZxStar_sstauhtaue",
        "label_H_ZxZxStar_sstauhtaum",
        "label_H_ZxZxStar_sstauhtauh",
        "label_H_ZxZxStar_stauhtaue",
        "label_H_ZxZxStar_stauhtaum",
        "label_H_ZxZxStar_stauhtauh",
        "label_H_ZxZxStar_qqee",
        "label_H_ZxZxStar_qqmm",
        "label_H_ZxZxStar_qqe",
        "label_H_ZxZxStar_qqm",
        "label_H_ZxZxStar_qee",
        "label_H_ZxZxStar_qmm",
        "label_H_ZxZxStar_qqtauhtaue",
        "label_H_ZxZxStar_qqtauhtaum",
        "label_H_ZxZxStar_qqtauhtauh",
        "label_H_ZxZxStar_qtauhtaue",
        "label_H_ZxZxStar_qtauhtaum",
        "label_H_ZxZxStar_qtauhtauh",
        "label_QCD_bb",
        "label_QCD_cc",
        "label_QCD_b",
        "label_QCD_c",
        "label_QCD_others",
    ]
    label_idx = {lab: i for i, lab in enumerate(label_list)}

    massmin = 30
    nbin, xmin, xmax = 22, 30, 250

    y_bkg_flag = " | ".join([f"(fj_label == {label_idx[lab]})" for lab in variable_dict[bname]])
    # y_score_sig = ' + '.join([f'score_{lab}' for lab in variable_dict[sname]])
    # y_score_bkg = ' + '.join([f'score_{lab}' for lab in variable_dict[bname]])
    # y_score_str = f'({y_score_sig}) / (({y_score_sig}) + ({y_score_bkg}))'

    ## a hack here ##!!
    name = taggername
    if name == "GloParT1":
        label = r"ParT"
        exist_variable_dict = {
            "QCD": " + ".join(
                [
                    f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probQCD{n}"
                    for n in ["b", "bb", "c", "cc", "others"]
                ]
            ),
            **{
                f"HIGGS2P{n}": f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}"
                for n in ["bb", "cc", "ss", "qq", "tauhtaue", "tauhtaum", "tauhtauh"]
            },
            **{
                f"HIG{n}": f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}"
                for n in ["bb", "cc", "ss", "qq", "tauhtaue", "tauhtaum", "tauhtauh"]
            },
            "HWWQQQQ": "pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}0c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}1c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}2c".format(
                n="WqqWqq"
            ),
            "HWWQQQ": "pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}0c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}1c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}2c".format(
                n="WqqWq"
            ),
            "HWWQQev": "pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}0c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}1c".format(
                n="WqqWev"
            ),
            "HWWQQmv": "pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}0c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}1c".format(
                n="WqqWmv"
            ),
            "HWWQQtauev": "pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}0c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}1c".format(
                n="WqqWtauev"
            ),
            "HWWQQtaumv": "pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}0c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}1c".format(
                n="WqqWtaumv"
            ),
            "HWWQQtauhv": "pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}0c + pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}1c".format(
                n="WqqWtauhv"
            ),
            "HWWQQta": " + ".join(
                [
                    f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probH{n}"
                    for n in [
                        "WqqWtauev0c",
                        "WqqWtauev1c",
                        "WqqWtaumv0c",
                        "WqqWtaumv1c",
                        "WqqWtauhv0c",
                        "WqqWtauhv1c",
                    ]
                ]
            ),
            "TTBARbWall": " + ".join(
                [
                    f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probTop{n}"
                    for n in [
                        "bWqq0c",
                        "bWqq1c",
                        "bWq0c",
                        "bWq1c",
                        "bWev",
                        "bWmv",
                        "bWtauev",
                        "bWtaumv",
                        "bWtauhv",
                    ]
                ]
            ),
            "TTBARbWhad": " + ".join(
                [
                    f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probTop{n}"
                    for n in ["bWqq0c", "bWqq1c", "bWq0c", "bWq1c"]
                ]
            ),
            "TTBARbWQQ": " + ".join(
                [
                    f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probTop{n}"
                    for n in ["bWqq0c", "bWqq1c"]
                ]
            ),
            "TTBARbWev": " + ".join(
                [f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probTop{n}" for n in ["bWev"]]
            ),
            "TTBARbWmv": " + ".join(
                [f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probTop{n}" for n in ["bWmv"]]
            ),
            "TTBARbWta": " + ".join(
                [
                    f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probTop{n}"
                    for n in ["bWtauev", "bWtaumv", "bWtauhv"]
                ]
            ),
            "TTBARbWtauhv": " + ".join(
                [
                    f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probTop{n}"
                    for n in ["bWtauhv"]
                ]
            ),
            "TTBARall": " + ".join(
                [
                    f"pfMassDecorrelatedInclParticleTransformerV1JetTags_probTop{n}"
                    for n in [
                        "bWqq0c",
                        "bWqq1c",
                        "bWq0c",
                        "bWq1c",
                        "bWev",
                        "bWmv",
                        "bWtauev",
                        "bWtaumv",
                        "bWtauhv",
                    ]
                ]
            ),
        }
        _nu = " + ".join([exist_variable_dict[s] for s in ["HWWQQQQ", "HWWQQQ"]])
        _de = " + ".join([exist_variable_dict[b] for b in ["QCD", "TTBARbWall"]])
        y_score_str = f"({_nu}) / (({_nu}) + ({_de}))"
    elif name == "DeepAK8MD":
        label = r"DeepAK8-MD (QCD veto)"
        exist_variable_dict = {
            "QCD": " + ".join(
                [
                    f"pfMassDecorrelatedDeepBoostedJetTags_probQCD{n}"
                    for n in ["b", "bb", "c", "cc", "others"]
                ]
            ),
            "TTBARbWall": " + ".join(
                [
                    f"pfMassDecorrelatedDeepBoostedJetTags_probT{n}"
                    for n in ["bc", "bq", "bcq", "bqq"]
                ]
            ),
            "HWWQQQQ": "pfMassDecorrelatedDeepBoostedJetTags_probHqqqq",
            "WQQ": "pfMassDecorrelatedDeepBoostedJetTags_probWcq + pfMassDecorrelatedDeepBoostedJetTags_probWqq",
        }
        ## a hack here ##!!
        _nu = " + ".join([exist_variable_dict[s] for s in ["HWWQQQQ", "WQQ"]])
        _de = " + ".join([exist_variable_dict[b] for b in ["QCD"]])
        y_score_str = f"({_nu}) / (({_nu}) + ({_de}))"
    elif name == "DeepAK8MDwTop":
        label = r"DeepAK8-MD"
        exist_variable_dict = {
            "QCD": " + ".join(
                [
                    f"pfMassDecorrelatedDeepBoostedJetTags_probQCD{n}"
                    for n in ["b", "bb", "c", "cc", "others"]
                ]
            ),
            "TTBARbWall": " + ".join(
                [
                    f"pfMassDecorrelatedDeepBoostedJetTags_probT{n}"
                    for n in ["bc", "bq", "bcq", "bqq"]
                ]
            ),
            "HWWQQQQ": "pfMassDecorrelatedDeepBoostedJetTags_probHqqqq",
            "WQQ": "pfMassDecorrelatedDeepBoostedJetTags_probWcq + pfMassDecorrelatedDeepBoostedJetTags_probWqq",
        }
        ## a hack here ##!!
        _nu = " + ".join([exist_variable_dict[s] for s in ["HWWQQQQ", "WQQ"]])
        _de = " + ".join([exist_variable_dict[b] for b in ["QCD", "TTBARbWall"]])
        y_score_str = f"({_nu}) / (({_nu}) + ({_de}))"

    # path = '/home/olympus/licq/hww/incl-train/weaver-core/weaver/predict/ak8_MD_inclv10beta4_ul_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.farm221.best_epoch/wda8'
    ###!!! On lxplus:
    # path = '/eos/user/c/coli/cms-repo/glopart-plotter/predict2/ak8_MD_inclv10beta4_ul_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.farm221.best_epoch/wda8'
    # on LPC
    path = "/ceph/cms/store/user/woodson/glopart-plotter/predict2/ak8_MD_inclv10beta4_ul_manual.nlayer10.vispart_as_resid.ddp4-bs640-lr1p2e-3.nepoch100.farm221.best_epoch/wda8"

    ###
    if sam == "qcd" or sam == "qcd-xsecw":
        filelist = glob.glob(f"{path}/pred_qcd*.root")
        # filelist = [f'{path}/pred_qcd300to470.root', f'{path}/pred_qcd470to600.root', f'{path}/pred_qcd600to800.root']#, f'{path}/pred_qcd800to1000.root', f'{path}/pred_qcd1000to1400.root', f'{path}/pred_qcd1400to1800.root', f'{path}/pred_qcd1800to2400.root', f'{path}/pred_qcd2400to3200.root', f'{path}/pred_qcd3200toinf.root']
    elif sam == "ttbar":
        filelist = glob.glob(f"{path}/pred_ttbar.root")
    print("Reading files...", filelist)
    df = uproot.lazy(filelist)
    df = df[ak.numexpr.evaluate(y_bkg_flag, df)]

    hists = {}
    for ptmin, ptmax in zip(pt_edges[:-1], pt_edges[1:]):
        df1 = df[
            ak.numexpr.evaluate(
                f"(fj_pt>{ptmin}) & (fj_pt<{ptmax}) & (abs(fj_eta)<2.4) & (fj_sdmass>{massmin})", df
            )
        ]
        print(y_bkg_flag)
        y_score = ak.numexpr.evaluate(y_score_str, df1)
        print(y_score_str)

        wps = [np.quantile(y_score, q=1 - t) for t in mistags]
        if mistags[0] == 1:
            wps[0] = 0.0
        print("WPs:", wps)

        for i, (mistag, wp) in enumerate(zip(mistags, wps)):
            df_t = df1[y_score > wp]
            print(len(df_t))
            # make histogram for sdmass
            hists[((ptmin, ptmax), mistag)] = bh.Histogram(
                bh.axis.Regular(nbin, xmin, xmax), storage=bh.storage.Weight()
            )
            hists[((ptmin, ptmax), mistag)].fill(df_t["fj_sdmass"])

            # content, yerr = hists[mistag].view().value, np.sqrt(hists[mistag].view().variance)

    return hists

In [ ]:
hists_ptbin = {}
for taggername in ["GloParT1", "DeepAK8MDwTop"]:
    hists_ptbin[taggername] = plot_mass_decorr_ptbinned(
        "qcd", taggername, "x", "QCD", pt_edges=[200, 400, 600, 1000, 1500, 2000, 3000]
    )

In [ ]:
hists_eb = {}
for taggername in ["GloParT1", "DeepAK8MDwTop"]:
    hists_eb[taggername] = plot_mass_decorr_ptbinned(
        "qcd",
        taggername,
        "x",
        "QCD",
        mistags=[1, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001],
        pt_edges=[600, 1000],
    )

In [ ]:
from scipy.stats import chi2


def binned_gof_pvalue(pred, obs, eps=1e-10):
    pred_norm = pred * np.sum(obs) / np.sum(pred)
    n2dll = 2 * np.sum(
        np.where(
            pred_norm > 0,
            pred_norm - obs + obs * np.log(obs / pred_norm + eps),
            np.zeros_like(pred_norm),
        )
    )
    # chi2val = np.sum(np.where(pred_norm > 0, (pred_norm - obs)**2 / np.maximum(obs, 1), np.zeros_like(pred_norm)))
    dof = np.count_nonzero(pred) - 1
    pval = 1 - chi2.cdf(n2dll, dof)
    return pval

### JSD vs jet pT

In [ ]:
# calculating JSDs
from scipy.spatial.distance import jensenshannon

nbin, xmin, xmax = 22, 30, 250
pt_edges = np.array([200, 400, 600, 1000, 1500, 2000, 3000])

jsds = {}
pvals = {}
for taggername in ["GloParT1", "DeepAK8MDwTop"]:
    for ptmin, ptmax in zip(pt_edges[:-1], pt_edges[1:]):
        f, ax = plt.subplots(figsize=(6, 6))
        norm_hval = {}
        raw_hval = {}
        for mistag in [1, 0.01]:
            _hist = hists_ptbin[taggername][((ptmin, ptmax), mistag)]
            content, yerr = _hist.view().value, np.sqrt(_hist.view().variance)
            raw_hval[mistag] = content
            content_norm, yerr_norm = content / np.sum(content), yerr / np.sum(
                content
            )  # normalize to 1
            norm_hval[mistag] = content_norm
            hep.histplot(
                content_norm,
                bins=_hist.axes[0].edges,
                yerr=yerr_norm,
                label=f"Mistag {mistag} (pt: {ptmin}-{ptmax})",
                ax=ax,
            )

        ax.set_xlabel("mSD", ha="right", x=1.0)
        ax.set_ylabel("Events / bins", ha="right", y=1.0)
        ax.legend()

        # calculating JSDs
        jsds[(taggername, (ptmin, ptmax))] = jensenshannon(norm_hval[1], norm_hval[0.01], base=base)
        pvals[(taggername, (ptmin, ptmax))] = binned_gof_pvalue(raw_hval[1], raw_hval[0.01])

In [ ]:
for taggername in ["GloParT1", "DeepAK8MDwTop"]:
    print(taggername)
    for ptmin, ptmax in zip(pt_edges[:-1], pt_edges[1:]):
        print((ptmin, ptmax))
        print("JSD", jsds[(taggername, (ptmin, ptmax))])
        print("GOF p-value", pvals[(taggername, (ptmin, ptmax))])

In [ ]:
from matplotlib.lines import Line2D

plot_kwargs_list = [
    dict(
        color="#5790fc",
        linestyle="none",
        marker="s",
        markersize=10,
        linewidth=3,
        markeredgecolor="#5790fc",
        markerfacecolor="white",
    ),
    dict(
        color="#f89c20",
        linestyle="none",
        marker="o",
        markersize=8,
        linewidth=3,
        markeredgecolor="#f89c20",
        markerfacecolor="#f89c20",
    ),
]
handle_kwargs_list = [
    dict(
        color="#5790fc",
        linestyle="-",
        marker="s",
        markersize=10,
        linewidth=3,
        markeredgecolor="#5790fc",
        markerfacecolor="white",
    ),
    dict(
        color="#f89c20",
        linestyle="-",
        marker="o",
        markersize=8,
        linewidth=3,
        markeredgecolor="#f89c20",
        markerfacecolor="#f89c20",
    ),
]
label_list = ["ParT", "DeepAK8-MD"]

f, ax = plt.subplots(figsize=(10, 10))
hep.cms.label(data=False, llabel="Simulation", loc=1, rlabel="(13 TeV)", year=2017, ax=ax)

handles = []
for i, taggername in enumerate(["GloParT1", "DeepAK8MDwTop"]):
    ax.errorbar(
        0.5 * (pt_edges[1:] + pt_edges[:-1]),
        [jsds[(taggername, (ptmin, ptmax))] for ptmin, ptmax in zip(pt_edges[:-1], pt_edges[1:])],
        xerr=0.5 * (pt_edges[1:] - pt_edges[:-1]),
        yerr=[0 for _ in pt_edges[1:]],
        label=label_list[i],
        **plot_kwargs_list[i],
    )

    handles.append(Line2D([0], [0], label=label_list[i], **handle_kwargs_list[i]))
ax.set_xlabel(r"$p_{T}$ [GeV]", ha="right", x=1.0)
ax.set_ylabel(r"JSD", ha="right", y=1.0)
ax.set_xlim(pt_edges[0], pt_edges[-1])
ax.set_ylim(7e-3 if base == 2 else 3e-3, 1)
ax.set_yscale("log")
ax.legend(loc="upper right", labelspacing=0.8, prop={"size": 24}, borderpad=0.8, handles=handles)
x = 0.04
ax.text(0.055, 0.78 + x, "QCD multijet", fontsize=24, transform=ax.transAxes)
ax.text(
    0.055, 0.71 + x, r"$200 < p_{T} < 3000$ GeV, $|\eta|<2.4$", fontsize=24, transform=ax.transAxes
)
ax.text(0.055, 0.64 + x, r"$30 < m_{SD} < 250$ GeV", fontsize=24, transform=ax.transAxes)
ax.text(0.055, 0.57 + x, r"$\epsilon_B = 1.0\%$", fontsize=24, transform=ax.transAxes)
plt.savefig(f"massdecorr_jsd_vs_pt_base{base}.pdf")

### JSD vs eB

In [ ]:
# calculating JSDs
from scipy.spatial.distance import jensenshannon

nbin, xmin, xmax = 22, 30, 250
pt_edges = np.array([600, 1000])

jsds = {}
pvals = {}
for taggername in ["GloParT1", "DeepAK8MDwTop"]:
    for ptmin, ptmax in zip(pt_edges[:-1], pt_edges[1:]):
        f, ax = plt.subplots(figsize=(6, 6))
        norm_hval = {}
        raw_hval = {}
        for mistag in [1, 0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
            _hist = hists_eb[taggername][((ptmin, ptmax), mistag)]
            content, yerr = _hist.view().value, np.sqrt(_hist.view().variance)
            raw_hval[mistag] = content
            content_norm, yerr_norm = content / np.sum(content), yerr / np.sum(
                content
            )  # normalize to 1
            norm_hval[mistag] = content_norm
            hep.histplot(
                content_norm,
                bins=_hist.axes[0].edges,
                yerr=yerr_norm,
                label=f"Mistag {mistag} (pt: {ptmin}-{ptmax})",
                ax=ax,
            )

        ax.set_xlabel("mSD", ha="right", x=1.0)
        ax.set_ylabel("Events / bins", ha="right", y=1.0)
        ax.legend(fontsize=10)

        # calculating JSDs
        for mistag in [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
            jsds[(taggername, (ptmin, ptmax), mistag)] = jensenshannon(
                norm_hval[1], norm_hval[mistag], base=base
            )
            pvals[(taggername, (ptmin, ptmax), mistag)] = binned_gof_pvalue(
                raw_hval[1], raw_hval[mistag]
            )

In [ ]:
from matplotlib.lines import Line2D

plot_kwargs_list = [
    dict(
        color="#5790fc",
        linestyle="none",
        marker="s",
        markersize=10,
        linewidth=3,
        markeredgecolor="#5790fc",
        markerfacecolor="white",
    ),
    dict(
        color="#f89c20",
        linestyle="none",
        marker="o",
        markersize=8,
        linewidth=3,
        markeredgecolor="#f89c20",
        markerfacecolor="#f89c20",
    ),
]
handle_kwargs_list = [
    dict(
        color="#5790fc",
        linestyle="-",
        marker="s",
        markersize=10,
        linewidth=3,
        markeredgecolor="#5790fc",
        markerfacecolor="white",
    ),
    dict(
        color="#f89c20",
        linestyle="-",
        marker="o",
        markersize=8,
        linewidth=3,
        markeredgecolor="#f89c20",
        markerfacecolor="#f89c20",
    ),
]
label_list = ["ParT", "DeepAK8-MD"]
ptmin, ptmax = 600, 1000
mistags = [0.1, 0.05, 0.02, 0.01, 0.005]

f, ax = plt.subplots(figsize=(10, 10))
hep.cms.label(data=False, llabel="Simulation", loc=1, rlabel="(13 TeV)", year=2017, ax=ax)

handles = []
for i, taggername in enumerate(["GloParT1", "DeepAK8MDwTop"]):
    ax.errorbar(
        np.arange(len(mistags)) + 0.5,
        [jsds[(taggername, (ptmin, ptmax), mistag)] for mistag in mistags],
        xerr=[0.5] * len(mistags),
        yerr=[0] * len(mistags),
        label=label_list[i],
        **plot_kwargs_list[i],
    )
    handles.append(Line2D([0], [0], label=label_list[i], **handle_kwargs_list[i]))

ax.set_xlabel(r"$\epsilon_B$ [%]", ha="right", x=1.0, labelpad=25)
ax.set_ylabel(r"JSD", ha="right", y=1.0)
ax.set_ylim(7e-3 if base == 2 else 3e-3, 1)
ax.set_yscale("log")
# ax.set_ylim(0, 0.12)
ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_xticks(np.arange(len(mistags)))
ax.set_xticklabels([""] * len(mistags))
ax.set_xlim(0, len(mistags))
tick_labels = [10.0, 5.0, 2.0, 1.0, 0.5]
for it in range(len(mistags)):
    ax.text(
        it + 0.5,
        -0.01,
        tick_labels[it],
        ha="center",
        va="top",
        transform=ax.get_xaxis_transform(),
        fontsize=22,
    )


ax.legend(loc="upper right", labelspacing=0.8, prop={"size": 24}, borderpad=0.8, handles=handles)


x = 0.04
ax.text(0.055, 0.78 + x, "QCD multijet", fontsize=24, transform=ax.transAxes)
ax.text(
    0.055, 0.71 + x, r"$600 < p_{T} < 1000$ GeV, $|\eta|<2.4$", fontsize=24, transform=ax.transAxes
)
ax.text(0.055, 0.64 + x, r"$30 < m_{SD} < 250$ GeV", fontsize=24, transform=ax.transAxes)
plt.savefig(f"massdecorr_jsd_vs_effbkg_base{base}.pdf")

In [ ]:
for taggername in ["GloParT1", "DeepAK8MDwTop"]:
    print(taggername)
    for ptmin, ptmax in zip(pt_edges[:-1], pt_edges[1:]):
        print((ptmin, ptmax))
        for mistag in [0.1, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
            print(mistag)
            print("JSD", jsds[(taggername, (ptmin, ptmax), mistag)])
            print("GOF p-value", pvals[(taggername, (ptmin, ptmax), mistag)])